# MWDAP Simulation: Distributed Algorithm vs. Centralized Baselines

This notebook compares three methods for the Minimum Weight Digraph
Augmentation Problem (MWDAP):

1. **Algorithm 1** — Distributed augmentation protocol (this paper)
2. **Frederickson–Ja'Ja'** — Centralized 2-approximation
3. **Exact ILP** — Multi-commodity flow formulation (for small instances)

Accompanies the paper:
> *Distributed graph augmentation protocols for weighted strong connectivity
> in multi-agent systems* (Ramos, Poças, Pequito)

**Requirements:** `networkx`, `numpy`, `scipy`

In [ ]:
import json
import numpy as np

from simulation import (
    generate_er_digraph, generate_euclidean_digraph, generate_dag_digraph,
    classify_sccs, distributed_algorithm, frederickson_jaja, exact_ilp,
    verify_augmentation, run_single, run_experiments,
)

## Experiment configuration

In [ ]:
# Configuration
ILP_MAX_N = 70       # ILP is exact but expensive; skip for large n
SAMPLES = 50         # samples per (n, model, parameter) tuple
NODE_SIZES = list(range(5, 101, 5))
MODELS = ['er', 'dag', 'euclidean']
PARAMS = {
    'er':        [lambda n: 1.0/n, lambda n: 2.0/n, lambda n: 3.0/n],
    'dag':       [lambda n: 2.0/n, lambda n: 4.0/n, lambda n: 6.0/n],
    'euclidean': [lambda n: 1.25/n**0.5, lambda n: 1.5/n**0.5, lambda n: 1.75/n**0.5],
}
SEED_BASE = 42

## Run experiments\n\nThis may take several hours for the full configuration above.

In [ ]:
all_model_results = {}

for model in MODELS:
    print(f'\n{"#"*60}')
    print(f'  GRAPH MODEL: {model.upper()}')
    print(f'{"#"*60}')
    
    results = run_experiments(
        NODE_SIZES,
        params=PARAMS[model],
        samples_per_size=SAMPLES,
        graph_model=model,
        ilp_max_n=ILP_MAX_N,
        seed_base=SEED_BASE,
    )
    all_model_results[model] = results

print('\nAll experiments completed.')

## Summary tables

In [ ]:
for model in MODELS:
    print(f'\n=== {model.upper()} ===')
    print(f'{"n":>4} | {"Dist/OPT":>10} | {"FJ/OPT":>10} | {"Dist/FJ":>10} | {"samples":>7}')
    print('-' * 60)
    for n in sorted(all_model_results[model].keys()):
        rlist = all_model_results[model][n]
        da_ilp = [r['ratio_da_ilp'] for r in rlist
                  if r.get('ratio_da_ilp') is not None]
        fj_ilp = [r['ratio_fj_ilp'] for r in rlist
                  if r.get('ratio_fj_ilp') is not None]
        da_fj  = [r['ratio_da_fj']  for r in rlist
                  if r.get('ratio_da_fj') is not None]
        print(f'{n:>4} | '
              f'{np.mean(da_ilp):>10.3f} | ' if da_ilp else f'{"":>10} | ',
              f'{np.mean(fj_ilp):>10.3f} | ' if fj_ilp else f'{"":>10} | ',
              f'{np.mean(da_fj):>10.3f} | ' if da_fj else f'{"":>10} | ',
              f'{len(rlist):>7}')

## Save results to JSON

In [ ]:
export = {}
for model, data in all_model_results.items():
    export[model] = {}
    for n, rlist in data.items():
        export[model][str(n)] = []
        for r in rlist:
            entry = {
                'alpha': r['alpha'],
                'beta': r['beta'],
                'd_in_max': r['d_in_max'],
                'da_cost': r['distributed']['cost'],
                'fj_cost': r['fj']['cost'],
                'ratio_da_ilp': r.get('ratio_da_ilp'),
                'ratio_fj_ilp': r.get('ratio_fj_ilp'),
                'ratio_da_fj': r.get('ratio_da_fj'),
            }
            if r.get('ilp') is not None and r['ilp']['cost'] < float('inf'):
                entry['ilp_cost'] = r['ilp']['cost']
            else:
                entry['ilp_cost'] = None
            export[model][str(n)].append(entry)

with open('mwdap_results.json', 'w') as f:
    json.dump(export, f, indent=2)
print('Results saved to mwdap_results.json')

## Empirical complexity estimation

Measures $(\\alpha + \\beta) d_{\\max}^{\\mathrm{in}}$ and fits a power law $O(|V|^x)$
to estimate the overall time complexity $O(|V|^{x+2})$.
This reproduces Table 1 from the paper.

In [ ]:
from scipy.optimize import curve_fit

def power_law(x, a, b):
    return a * np.power(x, b)

COMPLEXITY_SAMPLES = 30
COMPLEXITY_NODES = [10, 15, 20, 25, 30, 40, 50, 60, 70, 80, 90, 100]

table_rows = []

for model_name, gen_func, kwargs_fn in [
    ('ER', generate_er_digraph,
     lambda n: {'p': min(0.3, 3.0/n), 'weight_range': (1, 10)}),
    ('Euclidean', generate_euclidean_digraph,
     lambda n: {'radius': 0.15}),
    ('DAG', generate_dag_digraph,
     lambda n: {'p': min(0.4, 4.0/n), 'weight_range': (1, 10)}),
]:
    print(f'\n{"="*70}')
    print(f'  Model: {model_name}')
    print(f'{"="*70}')

    all_ns, all_ab_dmax = [], []

    for n in COMPLEXITY_NODES:
        ab_dmax_vals = []
        for s in range(COMPLEXITY_SAMPLES):
            seed = SEED_BASE + n * 1000 + s
            G, W = gen_func(n, seed=seed, **kwargs_fn(n))
            source_sccs, target_sccs, _, _, _ = classify_sccs(G)
            alpha = len(source_sccs)
            beta = len(target_sccs)
            d_in_max = max(dict(G.in_degree()).values())
            ab_dmax = (alpha + beta) * d_in_max
            ab_dmax_vals.append(ab_dmax)
            all_ns.append(n)
            all_ab_dmax.append(ab_dmax)

        print(f'  n={n:>3}: mean (α+β)d_max = {np.mean(ab_dmax_vals):.2f}')

    all_ns = np.array(all_ns, dtype=float)
    all_ab_dmax = np.array(all_ab_dmax, dtype=float)

    popt, _ = curve_fit(power_law, all_ns, all_ab_dmax, p0=[1, 1], maxfev=10000)
    exp_ab = popt[1]

    print(f'\n  (α+β)·d_max^in ≈ O(|V|^{exp_ab:.2f})')
    print(f'  Overall         ≈ O(|V|^{exp_ab + 2:.2f})')
    table_rows.append((model_name, exp_ab, exp_ab + 2))

print(f'\n{"="*70}')
print('  Summary (Table 1)')
print(f'{"="*70}')
print(f'  {"Model":<12} {"(α+β)d_max":>14} {"Overall":>14}')
print(f'  {"-"*42}')
for name, exp_ab, exp_ov in table_rows:
    print(f'  {name:<12} O(|V|^{exp_ab:.2f}){"":>5} O(|V|^{exp_ov:.2f})')

## TikZ/pgfplots export\n\nGenerates pgfplots-compatible TikZ code for inclusion in the paper.

In [ ]:
def generate_tikz(all_model_results):
    """Generate pgfplots TikZ code for all models."""
    
    for model in ['er', 'euclidean', 'dag']:
        data = all_model_results[model]
        ns = sorted(data.keys())
        
        stats = {}
        for n in ns:
            rlist = data[n]
            da_ilp = [r['ratio_da_ilp'] for r in rlist
                      if r.get('ratio_da_ilp') is not None]
            fj_ilp = [r['ratio_fj_ilp'] for r in rlist
                      if r.get('ratio_fj_ilp') is not None]
            da_cost = [r['distributed']['cost'] for r in rlist]
            fj_cost = [r['fj']['cost'] for r in rlist]
            ilp_cost = [r['ilp']['cost'] for r in rlist
                        if r.get('ilp') is not None
                        and r['ilp']['cost'] < float('inf')]
            stats[n] = {
                'da_ilp': (np.mean(da_ilp), np.std(da_ilp)) if da_ilp else (None, None),
                'fj_ilp': (np.mean(fj_ilp), np.std(fj_ilp)) if fj_ilp else (None, None),
                'da_cost': np.mean(da_cost),
                'fj_cost': np.mean(fj_cost),
                'ilp_cost': np.mean(ilp_cost) if ilp_cost else None,
            }
        
        label = {'er': 'ER', 'euclidean': 'Euclidean', 'dag': 'DAG'}[model]
        ns_ilp = [n for n in ns if stats[n]['da_ilp'][0] is not None]
        
        # --- (a) Approximation ratio ---
        print(f'% {label} Model: Approximation Ratio')
        print(r'\\begin{tikzpicture}')
        print(r'\\begin{axis}[')
        print(f'  xlabel={{Number of nodes ($n$)}},')
        print(r'  ylabel={Cost / OPT},')
        print(r'  legend pos=north west,')
        print(r'  grid=major, mark size=2pt,')
        print(r']')
        
        print(r'\\addplot[color=blue, mark=o, thick] coordinates {')
        for n in ns_ilp:
            m, _ = stats[n]['da_ilp']
            print(f'  ({n}, {m:.4f})')
        print(r'};')
        print(r'\\addlegendentry{Alg.\\ 1 / OPT}')
        
        print(r'\\addplot[color=red, mark=square, thick] coordinates {')
        for n in ns_ilp:
            m, _ = stats[n]['fj_ilp']
            print(f'  ({n}, {m:.4f})')
        print(r'};')
        print(r'\\addlegendentry{F-J / OPT}')
        
        print(r'\\end{axis}')
        print(r'\\end{tikzpicture}')
        print()
        
        # --- (b) Absolute cost ---
        print(f'% {label} Model: Absolute Cost')
        print(r'\\begin{tikzpicture}')
        print(r'\\begin{axis}[')
        print(f'  xlabel={{Number of nodes ($n$)}},')
        print(r'  ylabel={Absolute cost},')
        print(r'  legend pos=north west,')
        print(r'  grid=major, mark size=2pt,')
        print(r']')
        
        print(r'\\addplot[color=blue, mark=o, thick] coordinates {')
        for n in ns:
            print(f'  ({n}, {stats[n]["da_cost"]:.4f})')
        print(r'};')
        print(r'\\addlegendentry{Alg.\\ 1}')
        
        print(r'\\addplot[color=red, mark=square, thick] coordinates {')
        for n in ns:
            print(f'  ({n}, {stats[n]["fj_cost"]:.4f})')
        print(r'};')
        print(r'\\addlegendentry{F-J}')
        
        print(r'\\addplot[color=black, mark=triangle, thick] coordinates {')
        for n in ns_ilp:
            if stats[n]['ilp_cost'] is not None:
                print(f'  ({n}, {stats[n]["ilp_cost"]:.4f})')
        print(r'};')
        print(r'\\addlegendentry{OPT (ILP)}')
        
        print(r'\\end{axis}')
        print(r'\\end{tikzpicture}')
        print()

generate_tikz(all_model_results)